# TradeSense · Component 03 · Protocol B2 · step 3: XGBoost v2 (supervised pattern classifier)

All under the frozen Protocol B2 (stored in `layer3_v3/`). What this notebook does:
1. checks the protocol hash and that notebook 28 finished (it needs the AE-ctx score files); **TEST-B / TEST-3 are never loaded**;
2. builds the 53 rolling features (set A, same recipe as XGBoost v1) and the 8 AE-ctx columns (set B; the AE columns of the TRAIN-3 bars are **out-of-fold**);
3. unit tests;
4. selection on VAL-3 only: set {A, B} × scope {pooled, per_symbol} × depth {3, 5} = 8 configurations, **for each normalisation N0 and N1** (seed 11);
   - fit rows: TRAIN-3 only; early stopping and selection: VAL-3 only. The two new families (momentum_ignition_like, slow_accumulation_like) are **never used for fitting, early stopping or selection** (they stay unseen for H-B5);
   - selection metric: VAL-3 recall (macro over the 6 old families, mean over BTC/ETH) at the primary budget (2 episodes/day) with the deployable rolling threshold T1; tolerance 0.05 and simplest-first as in v1; H-B4 decides the feature set (set B only if it wins by at least 0.03);
5. final models (3 seeds: 11, 22, 33) for the selected configuration of each normalisation; scores of every series saved (genuine Layer 1, genuine Aug to 08-31, every TRAIN-3, VAL-3 and CAL-3 replica);
6. VAL-3 evaluation next to the comparators (AE-ctx fmax, robust-z) and a **chance baseline** (the model's own alarms placed at random times), leave-one-family-out (LOFO) for the 6 old families, unseen-family results, exact TreeSHAP;
7. share block.

What is **not** done here (so that no CAL information enters the selection): the CAL measurement (genuine alarm rates, CAL-3 recall), the N0/N1 adoption decision for XGBoost (H-B1/H-B2 on CAL) and the comparison with the v1 models. They are notebook 31.

CPU runtime. Every model is saved when it finishes; the notebook stops itself after `MAX_RUN_MINUTES` and continues on the next *Run all*.

In [ ]:
import os, json, glob, time, hashlib, warnings, sys, subprocess
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
try:
    import xgboost as xgb
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost']); import xgboost as xgb
from sklearn.metrics import average_precision_score
from scipy.stats import spearmanr
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L3 = BASE/'layer3_v3'; OUT = BASE/'models_b2_v1'; XG = OUT/'xgb'; XG.mkdir(parents=True, exist_ok=True)
SYMBOLS = ['BTCUSDT', 'ETHUSDT']; SEEDS = [11, 22, 33]; MODES = ['N0', 'N1']
MAX_RUN_MINUTES = float(os.environ.get('MAX_RUN_MINUTES', 150)); T_START = time.time(); NTHREAD = max(1, os.cpu_count() or 1)
LINES = []
def say(s=''): print(s); LINES.append(str(s))
def sha_file(p, n=16): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:n]
def out_of_time(): return (time.time() - T_START) / 60 > MAX_RUN_MINUTES
hf = L3/'protocol_bar_models_v2.sha256'; pf = L3/'protocol_bar_models_v2.md'
assert hf.exists() and pf.exists(), 'run notebook 27 first (it freezes the protocol)'
ph = hashlib.sha256(pf.read_text().encode()).hexdigest(); assert ph == hf.read_text().split()[0], 'frozen protocol was changed'
for s in SYMBOLS:
    for m in MODES: assert (OUT/'scores'/f'ae_ctx_{s}_{m}.npz').exists(), 'run notebook 28 to the end first (AE-ctx score files missing)'
say('protocol B2 intact, sha256 ' + ph[:16] + '; xgboost ' + xgb.__version__ + f'; {NTHREAD} threads')

Mounted at /content/drive
protocol B2 intact, sha256 443f99d5faf5929f; xgboost 3.4.1; 2 threads


## 1 · Library (same file as notebook 28)

In [ ]:
_old_lib = (OUT/'b2_lib.py').read_text() if (OUT/'b2_lib.py').exists() else None
open("b2_lib.py", "w").write(r'''
"""TradeSense C03 - Protocol B2 shared library: normalisation N0/N1, conditional autoencoder (AE-ctx), burden thresholds T0/T1, recall."""
import numpy as np, pandas as pd, torch, torch.nn as nn, time, json, hashlib
from numpy.lib.stride_tricks import sliding_window_view

FEATS = ['log_returns', 'trade_flow_imbalance', 'hl_spread_ratio', 'rush_order_density', 'volume_acceleration', 'large_trade_count']
LOGC = [3, 5]; WIN7 = 10080; MINP = 1440; CTX = 15; WARM = 30; GAPM = 5; CLIP = 20.0
BUDGETS = [1, 2, 4]; PRIMARY = 2; QGRID = [0.99, 0.993, 0.995, 0.997, 0.998, 0.999, 0.9993, 0.9995, 0.9997, 0.9998, 0.9999, 0.99995]
FAMS = {1: 'pump_and_dump_like', 2: 'fake_breakout_like', 3: 'abnormal_volume_burst', 4: 'trade_flow_imbalance', 5: 'wash_like_aggregate_proxy',
        6: 'liquidity_stress_proxy', 7: 'momentum_ignition_like', 8: 'slow_accumulation_like'}


# ---------------------------------------------------------------- normalisation
def to_x(F):
    x = np.asarray(F[FEATS] if hasattr(F, 'columns') else F, dtype='float64').copy()
    for j in LOGC: x[:, j] = np.log1p(x[:, j])
    return x


def fit_n0(F_train):
    """median / (IQR/1.349) on log1p-transformed Layer 1 TRAIN bars (caller passes bars >= 15 only)."""
    x = to_x(F_train); med = np.median(x, axis=0); q75, q25 = np.percentile(x, [75, 25], axis=0); return {'median': med, 'scale': (q75 - q25) / 1.349}


def normalise(F, mode, n0):
    x = to_x(F)
    if mode == 'N0': med, scl = n0['median'][None, :], n0['scale'][None, :]
    else:
        df = pd.DataFrame(x); r = df.rolling(WIN7, min_periods=MINP)
        med = r.median().shift(1).to_numpy(); q75 = r.quantile(0.75).shift(1).to_numpy(); q25 = r.quantile(0.25).shift(1).to_numpy()
        scl = np.maximum((q75 - q25) / 1.349, 0.1 * n0['scale'][None, :])
        bad = np.isnan(med) | np.isnan(scl); med = np.where(bad, n0['median'][None, :], med); scl = np.where(bad, n0['scale'][None, :], scl)
    return np.clip((x - med) / scl, -CLIP, CLIP)


def context(z):
    """causal context: mean and std (ddof 1) of the previous CTX bars; rows < CTX are zero-filled and never used."""
    n = len(z); m = np.zeros_like(z); s = np.zeros_like(z)
    w = sliding_window_view(z, CTX, axis=0)[: n - CTX]            # window k = bars k..k+14 -> bar k+15
    m[CTX:] = w.mean(axis=2); s[CTX:] = w.std(axis=2, ddof=1); return m, s


# ---------------------------------------------------------------- conditional autoencoder
class AECtx(nn.Module):
    def __init__(self, d=6, b=3):
        super().__init__(); self.enc = nn.Sequential(nn.Linear(d + 2 * d, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, b))
        self.dec = nn.Sequential(nn.Linear(b + 2 * d, 32), nn.ReLU(), nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, d))

    def forward(self, z, cm, cs):
        c = torch.cat([cm, cs], 1); return self.dec(torch.cat([self.enc(torch.cat([z, c], 1)), c], 1))


def train_ae(z, idx, seed, max_epochs=150, patience=10, batch=512, log=None):
    """z normalised genuine series; idx = bar indices allowed for fitting (valid, inside Layer 1 TRAIN). 90% fit / 10% early stop, 15-bar gap."""
    torch.manual_seed(seed); rng = np.random.default_rng(seed); cm, cs = context(z)
    idx = np.sort(idx); cut = int(0.9 * len(idx)); fit = idx[:cut]; val = idx[idx > idx[cut] + CTX]
    T = lambda a: torch.tensor(a, dtype=torch.float32)
    Zf, Mf, Sf = T(z[fit]), T(cm[fit]), T(cs[fit]); Zv, Mv, Sv = T(z[val]), T(cm[val]), T(cs[val])
    net = AECtx(); opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-5); best, bad, state = 1e18, 0, None
    for ep in range(max_epochs):
        net.train(); perm = torch.tensor(rng.permutation(len(fit)))
        for i in range(0, len(perm), batch):
            b = perm[i:i + batch]; opt.zero_grad(); loss = ((net(Zf[b], Mf[b], Sf[b]) - Zf[b]) ** 2).mean(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad(): v = float(((net(Zv, Mv, Sv) - Zv) ** 2).mean())
        if v < best - 1e-6: best, bad, state = v, 0, {k: x.clone() for k, x in net.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    net.load_state_dict(state); net.eval(); return net, {'epochs': ep + 1, 'best_val': best, 'fit_rows': int(len(fit)), 'val_rows': int(len(val)), 'fit_min': int(fit.min()), 'fit_max': int(fit.max())}


def ae_errors(net, z):
    cm, cs = context(z); T = lambda a: torch.tensor(a, dtype=torch.float32)
    with torch.no_grad(): rec = net(T(z), T(cm), T(cs)).numpy()
    return (z - rec) ** 2


def p99_scale(err, idx): return np.percentile(err[idx], 99, axis=0) + 1e-12


def ae_columns(err, scale):
    """8 columns: fmax score, fmean score, 6 per-feature errors / TRAIN p99 scale."""
    r = err / scale[None, :]; return np.column_stack([r.max(axis=1), err.mean(axis=1), r])


# ---------------------------------------------------------------- burden: episodes, thresholds, recall
def episodes(alarm, gap=GAPM):
    a = np.flatnonzero(alarm)
    if len(a) == 0: return 0
    return int(1 + np.sum(np.diff(a) > gap))


def eps_per_day(alarm, valid):
    a = alarm & valid; days = valid.sum() / 1440.0
    return episodes(a) / max(days, 1e-9)


def valid_mask(n, lo, hi): m = np.zeros(n, bool); m[max(lo, WARM):hi] = True; return m


def t0_threshold(score, valid, budget):
    """lowest quantile-grid threshold (descending probability grid, lowering the threshold step by step) with episodes/day <= budget."""
    s = score[valid]; best = None
    for q in sorted(QGRID, reverse=True):
        thr = np.quantile(s, q)
        if eps_per_day(score > thr, valid) <= budget: best = thr
        else: break
    return best if best is not None else np.quantile(s, max(QGRID))


def rolling_thr(score, q):
    return pd.Series(score).rolling(WIN7, min_periods=MINP).quantile(q).shift(1).to_numpy()


def t1_q(score, valid, budget):
    """smallest quantile level q (lowest rolling threshold) with episodes/day <= budget on the genuine DEV bars."""
    for q in QGRID:
        thr = rolling_thr(score, q); alarm = (score > np.where(np.isnan(thr), np.inf, thr))
        if eps_per_day(alarm, valid) <= budget: return q
    return QGRID[-1]


def alarms(score, rule, par):
    if rule == 'T0': return score > par
    thr = rolling_thr(score, par); return score > np.where(np.isnan(thr), np.inf, thr)


def recall_by_family(alarm, ev, seg_ok=None):
    """ev rows: t0, dur, family, (replica); detected if an alarm falls in [t0, t0+dur-1]."""
    c = np.concatenate([[0], np.cumsum(alarm.astype(np.int64))]); out = {}
    for fam, sub in ev.groupby('family'):
        hit = [(c[r.t0 + r.dur] - c[r.t0]) > 0 for r in sub.itertuples()]; out[int(fam)] = (int(np.sum(hit)), int(len(hit)))
    return out


def bootstrap_ci(hits, n=1000, seed=0):
    rng = np.random.default_rng(seed); h = np.asarray(hits, float)
    if len(h) == 0: return (np.nan, np.nan)
    b = [h[rng.integers(0, len(h), len(h))].mean() for _ in range(n)]; return (float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5)))


def macro(rec, fams):
    v = [rec[f][0] / rec[f][1] for f in fams if f in rec and rec[f][1] > 0]; return float(np.mean(v)) if v else float('nan')


def sha_bytes(b): return hashlib.sha256(b).hexdigest()

''')

import sys, importlib; sys.path.insert(0, '.'); import b2_lib as B; importlib.reload(B); (OUT/'b2_lib.py').write_text(Path('b2_lib.py').read_text())
say('library sha256 ' + sha_file('b2_lib.py'))
assert _old_lib is None or _old_lib.strip() == Path('b2_lib.py').read_text().strip(), 'b2_lib.py differs from the copy written by notebook 28'

library sha256 1b0b46630763c944


## 2 · Data, features (set A and set B) and row masks

In [ ]:
LOOK = 15; FEATS = B.FEATS; BUDGETS = B.BUDGETS; PB = B.PRIMARY; OLD, NEW = [1, 2, 3, 4, 5, 6], [7, 8]
def find1(pat):
    c = [x for x in glob.glob(str(BASE/'**'/pat), recursive=True) if 'backup' not in x.lower()]; assert c, pat; return sorted(c)[0]
D = {}; SCO = {}
for s in SYMBOLS:
    a = pd.read_parquet(find1(f'{s.lower()}_l1_feature_matrix_90d.parquet')); a = a.set_index('datetime') if 'datetime' in a.columns else a
    b = pd.read_parquet(BASE/'layer1_aug_oct_v1'/f'{s.lower()}_l1_feature_matrix_aug_oct_2024.parquet').iloc[:31*1440]      # genuine Aug only (to 2024-08-31): TEST-B is never loaded
    for d in (a, b): d.loc[d.index[:15], 'volume_acceleration'] = 0.0
    sets = {}
    for name in ['TRAIN', 'VAL', 'CAL']:
        f = pd.read_parquet(L3/f'l3v3_{s}_{name}.parquet'); ev = pd.read_csv(L3/f'l3v3_{s}_{name}_events.csv')
        sets[name] = {int(r): (g.reset_index(drop=True), ev[ev.replica == r].reset_index(drop=True)) for r, g in f.groupby('replica')}
    D[s] = {'GEN_L1': a[FEATS].reset_index(drop=True), 'GEN_AO': b[FEATS].reset_index(drop=True), 'sets': sets}
    for mode in MODES:
        z = np.load(OUT/'scores'/f'ae_ctx_{s}_{mode}.npz'); SCO[(s, mode)] = {k: z[k] for k in z.files if k != 'scale'}
    for mode in MODES:
        for k, v in SCO[(s, mode)].items():
            n_ = len(D[s]['GEN_L1']) if k == 'GEN_L1' else len(D[s]['GEN_AO']) if k == 'GEN_AO' else len(D[s]['sets'][k.split('_r')[0]][int(k.split('_r')[1])][0])
            assert len(v) == n_ and v.shape[1] == 8, (s, mode, k)
    say(f'{s}: genuine Layer 1 {len(a)} bars, genuine Aug {len(b)} bars; replicas ' + str({k: len(v) for k, v in sets.items()}))
assert not any('TEST' in k for s in SYMBOLS for m in MODES for k in SCO[(s, m)])
N0 = {s: B.fit_n0(D[s]['GEN_L1'].iloc[15:90720]) for s in SYMBOLS}

def build_feats(Z):
    """53 causal rolling features of the six normalised features (identical recipe to XGBoost v1, look-back <= 15 bars)."""
    df = pd.DataFrame(np.asarray(Z, np.float64), columns=FEATS); cols = {}; prev15 = df.shift(1).rolling(15).mean()
    for f in FEATS:
        s_ = df[f]; a_ = s_.abs()
        cols[f + '__z'] = s_; cols[f + '__d1'] = s_ - s_.shift(1); cols[f + '__mean3'] = s_.rolling(3).mean(); cols[f + '__mean15'] = s_.rolling(15).mean(); cols[f + '__amax3'] = a_.rolling(3).max()
        cols[f + '__amax15'] = a_.rolling(15).max(); cols[f + '__std15'] = s_.rolling(15).std(); cols[f + '__dev'] = s_ - prev15[f]
    A = df.abs(); ne = (A > 3).sum(axis=1).astype(float); sa = A.sum(axis=1)
    cols['x__n_extreme'] = ne; cols['x__sumabs'] = sa; cols['x__n_extreme_mean5'] = ne.rolling(5).mean(); cols['x__sumabs_mean5'] = sa.rolling(5).mean(); cols['x__sumabs_max15'] = sa.rolling(15).max()
    out = pd.DataFrame(cols).to_numpy(np.float32, copy=True); out[:B.WARM] = np.nan; return out, list(cols)
FA_NAMES = build_feats(np.zeros((200, 6)))[1]
NAMES = {'A': FA_NAMES, 'B': FA_NAMES + ['ae__fmax', 'ae__fmean'] + ['ae__e_' + f for f in FEATS]}; NCOL = {'A': len(FA_NAMES), 'B': len(FA_NAMES) + 8}
assert NCOL == {'A': 53, 'B': 61}
XS = {}; RZ = {}
def series_F(s, key):
    if key in ('GEN_L1', 'GEN_AO'): return D[s][key]
    name, r = key.split('_r'); return D[s]['sets'][name][int(r)][0]
def XOF(s, mode, key, cache=True):
    k = (s, mode, key)
    if k in XS: return XS[k]
    z = B.normalise(series_F(s, key), mode, N0[s]); X = np.hstack([build_feats(z)[0], SCO[(s, mode)][key].astype(np.float32)]); X[:B.WARM] = np.nan
    if key.startswith('VAL') or key == 'GEN_L1': RZ[k] = np.abs(z).max(axis=1).astype(np.float32)
    if cache and not key.startswith(('CAL', 'GEN_AO')): XS[k] = X
    return X
# row masks of the TRAIN-3 and VAL-3 replicas (label: 0 clean, 1..8 family, -1 tail)
RS = {}
for s in SYMBOLS:
    for name in ['TRAIN', 'VAL']:
        for r, (f, ev) in D[s]['sets'][name].items():
            lab = f.label.to_numpy().astype(int); inseg = f.in_seg.to_numpy().astype(bool); n = len(f); idx = np.arange(n)
            c = np.concatenate([[0], np.cumsum((lab != 0).astype(np.int64))])
            clean = ((c[idx + 1] - c[np.maximum(idx - LOOK, 0)]) == 0) & inseg & (idx >= B.WARM)      # no non-clean bar in [t-15, t]
            pos = (lab > 0) & inseg & (idx >= B.WARM); assert not (clean & pos).any(); RS[(s, name, r)] = (lab, inseg, clean, pos)
say('rows: ' + str({(s, n_): int(sum(RS[(s, n_, r)][3].sum() for r in D[s]['sets'][n_])) for s in SYMBOLS for n_ in ['TRAIN', 'VAL']}) + ' positive bars; clean negatives (replica 0) ' + str({(s, n_): int(RS[(s, n_, 0)][2].sum()) for s in SYMBOLS for n_ in ['TRAIN', 'VAL']}))

BTCUSDT: genuine Layer 1 129600 bars, genuine Aug 44640 bars; replicas {'TRAIN': 2, 'VAL': 2, 'CAL': 1}
ETHUSDT: genuine Layer 1 129600 bars, genuine Aug 44640 bars; replicas {'TRAIN': 2, 'VAL': 2, 'CAL': 1}
rows: {('BTCUSDT', 'TRAIN'): 11953, ('BTCUSDT', 'VAL'): 8432, ('ETHUSDT', 'TRAIN'): 11917, ('ETHUSDT', 'VAL'): 8368} positive bars; clean negatives (replica 0) {('BTCUSDT', 'TRAIN'): 75762, ('BTCUSDT', 'VAL'): 30230, ('ETHUSDT', 'TRAIN'): 75874, ('ETHUSDT', 'VAL'): 30085}


## 3 · Unit tests

In [ ]:
res = {}
s0 = SYMBOLS[0]
# T01 clean-window features are identical in the genuine and the injected series (static normalisation N0)
f0 = D[s0]['sets']['TRAIN'][0][0]; lab, inseg, clean, pos = RS[(s0, 'TRAIN', 0)]
XA_g = build_feats(B.normalise(D[s0]['GEN_L1'], 'N0', N0[s0]))[0]; XA_i = build_feats(B.normalise(f0, 'N0', N0[s0]))[0]
ii = np.flatnonzero(clean); res['T01 features of clean negative windows equal the genuine series (N0)'] = bool(np.allclose(XA_g[ii], XA_i[ii], rtol=1e-3, atol=1e-3, equal_nan=True))
# T02 features causal
z = B.normalise(D[s0]['GEN_L1'].iloc[:5000], 'N0', N0[s0]); z2 = z.copy(); z2[3000:] *= 4
res['T02 rolling features are causal (changing bars after t leaves values <= t unchanged)'] = bool(np.allclose(build_feats(z)[0][:3000], build_feats(z2)[0][:3000], equal_nan=True))
# T03 a plain recompute of three features
t = 4000; w = z[t - 15:t + 1]; Xa = build_feats(z)[0]; j = FA_NAMES.index('log_returns__mean3'); k = FA_NAMES.index('volume_acceleration__std15'); l_ = FA_NAMES.index('large_trade_count__dev')
res['T03 feature values equal a plain recompute'] = bool(np.isclose(Xa[t, j], z[t - 2:t + 1, 0].mean(), atol=1e-5) and np.isclose(Xa[t, k], z[t - 14:t + 1, 4].std(ddof=1), atol=1e-4) and np.isclose(Xa[t, l_], z[t, 5] - z[t - 15:t, 5].mean(), atol=1e-4))
# T04 recall rule against a plain loop
def event_hits(alarm, ev):
    c = np.concatenate([[0], np.cumsum(alarm.astype(np.int64))]); t0 = ev.t0.to_numpy(); du = ev.dur.to_numpy(); return (c[t0 + du] - c[t0]) > 0
rng = np.random.default_rng(0); al = rng.random(5000) < 0.01; evt = pd.DataFrame({'t0': rng.integers(100, 4000, 80), 'dur': rng.integers(1, 40, 80)})
plain = np.array([bool(al[r.t0:r.t0 + r.dur].any()) for r in evt.itertuples()]); res['T04 event detection equals a plain loop'] = bool((event_hits(al, evt) == plain).all())
# T05 SCO columns: finite for bars >= 30, one row per bar
res['T05 AE columns exist and are finite for bars >= 30'] = all(np.isfinite(SCO[(s, m)][k][B.WARM:]).all() for s in SYMBOLS for m in MODES for k in SCO[(s, m)])
# T06 same seed, same booster scores (small model)
Xs = XOF(s0, 'N0', 'TRAIN_r0')[:, :53]; ys = (RS[(s0, 'TRAIN', 0)][3]).astype(np.float32); mm = ~np.isnan(Xs).any(axis=1) & (RS[(s0, 'TRAIN', 0)][3] | RS[(s0, 'TRAIN', 0)][2])
dm = xgb.DMatrix(Xs[mm][:20000], label=ys[mm][:20000]); pr = {'objective': 'binary:logistic', 'tree_method': 'hist', 'max_depth': 3, 'eta': 0.1, 'seed': 11, 'nthread': NTHREAD}
p1 = xgb.train(pr, dm, 40).predict(dm, output_margin=True); p2 = xgb.train(pr, dm, 40).predict(dm, output_margin=True)
res['T06 same seed reproduces the booster scores (Spearman >= 0.98)'] = bool(spearmanr(p1, p2)[0] >= 0.98)
# T07 chance baseline limits
def chance_recall(alarm_dev, evs, fams, nsh=100, seed=0):
    rg = np.random.default_rng(seed); out = []
    for _ in range(nsh):
        full = np.zeros(129600, bool); full[90720:] = np.roll(alarm_dev, int(rg.integers(0, len(alarm_dev)))); h = event_hits(full, evs)
        d_ = pd.DataFrame({'fam': evs.family.to_numpy(), 'h': h}); d_ = d_[d_.fam.isin(fams)]; out.append(d_.groupby('fam').h.mean().mean())
    return float(np.mean(out))
evt2 = pd.DataFrame({'t0': rng.integers(90800, 129000, 60), 'dur': rng.integers(1, 30, 60), 'family': rng.integers(1, 7, 60)})
res['T07 chance baseline: no alarms -> 0, alarms everywhere -> 1'] = bool(chance_recall(np.zeros(38880, bool), evt2, OLD, 5) == 0.0 and chance_recall(np.ones(38880, bool), evt2, OLD, 5) == 1.0)
for k_, v in res.items(): say(('PASS ' if v else 'FAIL ') + k_)
assert all(res.values())

PASS T01 features of clean negative windows equal the genuine series (N0)
PASS T02 rolling features are causal (changing bars after t leaves values <= t unchanged)
PASS T03 feature values equal a plain recompute
PASS T04 event detection equals a plain loop
PASS T05 AE columns exist and are finite for bars >= 30
PASS T06 same seed reproduces the booster scores (Spearman >= 0.98)
PASS T07 chance baseline: no alarms -> 0, alarms everywhere -> 1


## 4 · Selection on VAL-3 (8 configurations × N0/N1)

In [ ]:
PARAMS = {'objective': 'binary:logistic', 'eval_metric': 'aucpr', 'tree_method': 'hist', 'eta': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 5, 'reg_lambda': 1.0, 'scale_pos_weight': 5.0, 'nthread': NTHREAD}
MAX_ROUNDS, ES_ROUNDS = 800, 40
ROWLOG = []
def get_rows(s, mode, name, fs, lofo=None, tag=''):
    """positives from every replica; clean negatives from replica 0 only (the genuine bars are the same in all replicas). VAL: only the 6 old families; LOFO: one more family removed."""
    Xs, ys = [], []
    for r, (f, ev) in D[s]['sets'][name].items():
        lab, inseg, clean, pos = RS[(s, name, r)]; p = pos.copy()
        if name == 'VAL': p &= (lab <= 6)
        if lofo is not None: p &= (lab != lofo)
        m = p | (clean if r == 0 else np.zeros(len(lab), bool)); ii = np.flatnonzero(m)
        ROWLOG.append({'tag': tag, 'sym': s, 'set': name, 'min_bar': int(ii.min()), 'max_bar': int(ii.max()), 'n': int(len(ii)), 'n_pos': int(p[ii].sum()), 'pos_labels': sorted(set(lab[ii][p[ii]].tolist())), 'outside_segment': int((~inseg[ii]).sum())})
        Xs.append(XOF(s, mode, f'{name}_r{r}')[ii][:, :NCOL[fs]]); ys.append(p[ii].astype(np.float32))
    return np.concatenate(Xs), np.concatenate(ys)
def bpath(mode, fs, scope, key, depth, seed, lofo=None): return XG/(f'{mode}_{fs}_{scope}_{key}_d{depth}_s{seed}' + ('' if lofo is None else f'_no{lofo}') + '.json')
def train_one(mode, fs, scope, key, syms, depth, seed, lofo=None):
    p = bpath(mode, fs, scope, key, depth, seed, lofo); ip = Path(str(p)[:-5] + '.info.json')
    if p.exists() and ip.exists(): return False
    tag = f'{mode}|{fs}|{scope}|{key}|d{depth}|s{seed}|no{lofo}'; n0_ = len(ROWLOG)
    parts = [get_rows(s, mode, 'TRAIN', fs, lofo, tag + '|fit') for s in syms]; ves = [get_rows(s, mode, 'VAL', fs, lofo, tag + '|es') for s in syms]
    Xt = np.concatenate([x[0] for x in parts]); yt = np.concatenate([x[1] for x in parts]); Xv = np.concatenate([x[0] for x in ves]); yv = np.concatenate([x[1] for x in ves])
    dtr = xgb.DMatrix(Xt, label=yt, feature_names=NAMES[fs]); dva = xgb.DMatrix(Xv, label=yv, feature_names=NAMES[fs]); t = time.time()
    bst = xgb.train({**PARAMS, 'max_depth': depth, 'seed': seed}, dtr, num_boost_round=MAX_ROUNDS, evals=[(dva, 'val')], early_stopping_rounds=ES_ROUNDS, verbose_eval=False)
    bst.save_model(str(p)); info = {'best_iteration': int(bst.best_iteration), 'best_aucpr_val': float(bst.best_score), 'n_fit': int(len(yt)), 'n_fit_pos': int(yt.sum()), 'n_es': int(len(yv)), 'n_es_pos': int(yv.sum()), 'seconds': round(time.time() - t, 1), 'rows': ROWLOG[n0_:]}
    ip.write_text(json.dumps(info)); say(f'  trained {p.stem}: best iteration {info["best_iteration"]}, fit {len(yt):,} rows ({int(yt.sum()):,} positive), {info["seconds"]}s'); return True

class Model:
    def __init__(self, mode, fs, scope, depth, seeds, lofo=None): self.mode, self.fs, self.scope, self.depth, self.seeds, self.lofo = mode, fs, scope, depth, list(seeds), lofo; self.cache = {}; self.bst = {}
    def boosters(self, sym):
        key = 'pool' if self.scope == 'pooled' else sym
        if key not in self.bst:
            out = []
            for sd in self.seeds:
                p = bpath(self.mode, self.fs, self.scope, key, self.depth, sd, self.lofo); b_ = xgb.Booster(); b_.load_model(str(p)); out.append((b_, json.loads(Path(str(p)[:-5] + '.info.json').read_text())['best_iteration']))
            self.bst[key] = out
        return self.bst[key]
    def score(self, sym, key):
        if (sym, key) not in self.cache:
            dm = xgb.DMatrix(XOF(sym, self.mode, key)[:, :NCOL[self.fs]], feature_names=NAMES[self.fs])
            m = np.mean([b_.predict(dm, output_margin=True, iteration_range=(0, bi + 1)) for b_, bi in self.boosters(sym)], axis=0).astype(np.float32); m[:B.WARM] = np.nan; self.cache[(sym, key)] = m
        return self.cache[(sym, key)]

L1V = B.valid_mask(129600, 90720, 129600)
def val_hits(get_score, rule, bud):
    recs = []; dev = {}
    for s in SYMBOLS:
        g = get_score(s, 'GEN_L1').astype('float64'); par = B.t0_threshold(g, L1V, bud) if rule == 'T0' else B.t1_q(g, L1V, bud); dev[s] = B.eps_per_day(B.alarms(g, rule, par), L1V)
        for r, (f, ev) in D[s]['sets']['VAL'].items():
            al = B.alarms(get_score(s, f'VAL_r{r}').astype('float64'), rule, par); recs.append(pd.DataFrame({'sym': s, 'fam': ev.family.to_numpy(), 'hit': event_hits(al, ev)}))
    return pd.concat(recs, ignore_index=True), dev
def macro_rate(dfh, fams):
    return float(np.mean([dfh[(dfh.sym == s) & dfh.fam.isin(fams)].groupby('fam').hit.mean().mean() for s in SYMBOLS]))
def pr_auc_val(model):
    v = []
    for s in SYMBOLS:
        ys, ss = [], []
        for r in D[s]['sets']['VAL']:
            lab, inseg, clean, pos = RS[(s, 'VAL', r)]; p = pos & (lab <= 6); m = p | (clean if r == 0 else np.zeros(len(lab), bool)); ys.append(p[m]); ss.append(model.score(s, f'VAL_r{r}')[m])
        v.append(average_precision_score(np.concatenate(ys), np.concatenate(ss)))
    return float(np.mean(v))

CONFIGS = [(fs, scope, depth) for fs in ['A', 'B'] for scope in ['pooled', 'per_symbol'] for depth in [3, 5]]
def cname(c): return f'{c[0]}|{c[1]}|d{c[2]}'
def train_cfg(mode, c, seed, lofo=None):
    fs, scope, depth = c
    if scope == 'pooled': return train_one(mode, fs, scope, 'pool', SYMBOLS, depth, seed, lofo)
    return any([train_one(mode, fs, scope, s, [s], depth, seed, lofo) for s in SYMBOLS])
SEL_DONE = True; SELF = {}
say('--- selection on VAL-3 (seed 11); metric = recall macro over the 6 old families, mean BTC/ETH, T1, 2 episodes/day ---')
for mode in MODES:
    for c in CONFIGS:
        jf = XG/f'sel_{mode}_{cname(c).replace("|", "_")}.json'
        if jf.exists(): SELF[(mode, c)] = json.loads(jf.read_text()); continue
        if out_of_time(): SEL_DONE = False; continue
        train_cfg(mode, c, 11); m = Model(mode, c[0], c[1], c[2], [11])
        h1, dev1 = val_hits(m.score, 'T1', PB); h0, dev0 = val_hits(m.score, 'T0', PB)
        r = {'mode': mode, 'config': cname(c), 'fs': c[0], 'scope': c[1], 'depth': c[2], 'metric': macro_rate(h1, OLD), 'metric_T0': macro_rate(h0, OLD), 'recall_new2_T1': macro_rate(h1, NEW), 'pr_auc': pr_auc_val(m),
             'dev_eps_T1': float(np.mean(list(dev1.values()))), 'dev_eps_T0': float(np.mean(list(dev0.values())))}
        jf.write_text(json.dumps(r)); SELF[(mode, c)] = r; say(f'{mode} {cname(c):18s} VAL-3 recall(old6,T1) {r["metric"]:.3f}  T0 {r["metric_T0"]:.3f}  PR-AUC {r["pr_auc"]:.3f}  DEV eps/day {r["dev_eps_T1"]:.2f}')
if not SEL_DONE: say('>>> time budget reached: Run all again to continue (finished models are kept). The rest is skipped until all 16 configurations are done. <<<')

--- selection on VAL-3 (seed 11); metric = recall macro over the 6 old families, mean BTC/ETH, T1, 2 episodes/day ---
  trained N0_A_pooled_pool_d3_s11: best iteration 799, fit 175,506 rows (23,870 positive), 24.0s
N0 A|pooled|d3        VAL-3 recall(old6,T1) 0.134  T0 0.689  PR-AUC 0.797  DEV eps/day 1.50
  trained N0_A_pooled_pool_d5_s11: best iteration 666, fit 175,506 rows (23,870 positive), 28.2s
N0 A|pooled|d5        VAL-3 recall(old6,T1) 0.149  T0 0.739  PR-AUC 0.813  DEV eps/day 1.63
  trained N0_A_per_symbol_BTCUSDT_d3_s11: best iteration 799, fit 87,715 rows (11,953 positive), 12.9s
  trained N0_A_per_symbol_ETHUSDT_d3_s11: best iteration 794, fit 87,791 rows (11,917 positive), 12.8s
N0 A|per_symbol|d3    VAL-3 recall(old6,T1) 0.132  T0 0.691  PR-AUC 0.797  DEV eps/day 1.56
  trained N0_A_per_symbol_BTCUSDT_d5_s11: best iteration 685, fit 87,715 rows (11,953 positive), 15.1s
  trained N0_A_per_symbol_ETHUSDT_d5_s11: best iteration 486, fit 87,791 rows (11,917 positive), 11.4s


## 5 · Selection rule (H-B4)

In [ ]:
SELECTED = {}; HB4 = {}
if SEL_DONE:
    rows = [r for (m, c), r in SELF.items()]; TAB = pd.DataFrame(rows); TAB.to_csv(OUT/'xgb_selection_table.csv', index=False)
    def decide(tab_mode):
        bestA = tab_mode[tab_mode.fs == 'A'].metric.max(); bestB = tab_mode[tab_mode.fs == 'B'].metric.max(); hb4 = bool(bestB - bestA >= 0.03); allowed = 'B' if hb4 else 'A'
        t = tab_mode[tab_mode.fs == allowed]; best = t.metric.max(); tied = t[t.metric >= best - 0.05].copy()
        tied['k'] = list(zip((tied.scope != 'pooled').astype(int), tied.depth, -tied.pr_auc)); sel = tied.sort_values('k').iloc[0]
        return {'best_A': float(bestA), 'best_B': float(bestB), 'H-B4 (set B beats set A by >= 0.03)': hb4, 'allowed_set': allowed, 'selected': sel.config}
    for mode in MODES:
        d_ = decide(TAB[TAB['mode'] == mode]); HB4[mode] = d_; SELECTED[mode] = tuple([d_['selected'].split('|')[0], d_['selected'].split('|')[1], int(d_['selected'].split('|')[2][1:])])
        say(f'{mode}: best set A {d_["best_A"]:.3f}, best set B {d_["best_B"]:.3f} -> H-B4 {d_["H-B4 (set B beats set A by >= 0.03)"]}; selected {d_["selected"]}')
    # recompute the rule from the saved csv alone
    TAB2 = pd.read_csv(OUT/'xgb_selection_table.csv'); assert all(decide(TAB2[TAB2['mode'] == m_])['selected'] == HB4[m_]['selected'] for m_ in MODES) and len(TAB2) == 16
    say('PASS T08 the selection is reproduced from the saved table alone (16 configurations)')

N0: best set A 0.149, best set B 0.132 -> H-B4 False; selected A|pooled|d3
N1: best set A 0.145, best set B 0.143 -> H-B4 False; selected A|pooled|d3
PASS T08 the selection is reproduced from the saved table alone (16 configurations)


## 6 · Final seeds and leave-one-family-out models

In [ ]:
FIN_DONE = SEL_DONE
if SEL_DONE:
    say('--- final seeds and leave-one-family-out models for the selected configurations ---')
    for mode in MODES:
        c = SELECTED[mode]
        for sd in SEEDS[1:]:
            if out_of_time(): FIN_DONE = False; break
            train_cfg(mode, c, sd)
        for fam in OLD:
            if out_of_time(): FIN_DONE = False; break
            train_cfg(mode, c, 11, lofo=fam)
    if not FIN_DONE: say('>>> time budget reached: Run all again to continue. <<<')
    else:
        # T09: no row of an excluded family in the LOFO fit and early-stopping sets; fit rows inside TRAIN-3, early-stopping rows inside VAL-3 and without the new families
        ok = True
        for p in XG.glob('*.info.json'):
            inf = json.loads(p.read_text())
            for r in inf['rows']:
                ok &= r['outside_segment'] == 0
                if r['tag'].endswith('|fit'): ok &= r['max_bar'] < 90720 and r['min_bar'] >= B.WARM and not any(l > 6 for l in r['pos_labels'])
                if r['tag'].endswith('|es'): ok &= r['min_bar'] >= 90720 and r['max_bar'] < 129600 and not any(l > 6 for l in r['pos_labels'])
                if '|no' in r['tag'] and not r['tag'].split('|no')[1].split('|')[0] == 'None': ok &= int(r['tag'].split('|no')[1].split('|')[0]) not in r['pos_labels']
        say(('PASS' if ok else 'FAIL') + ' T09 fit rows inside TRAIN-3, early-stopping rows inside VAL-3, new families never used, LOFO family removed'); assert ok

--- final seeds and leave-one-family-out models for the selected configurations ---
  trained N0_A_pooled_pool_d3_s22: best iteration 799, fit 175,506 rows (23,870 positive), 24.5s
  trained N0_A_pooled_pool_d3_s33: best iteration 797, fit 175,506 rows (23,870 positive), 24.8s
  trained N0_A_pooled_pool_d3_s11_no1: best iteration 796, fit 171,686 rows (20,050 positive), 22.3s
  trained N0_A_pooled_pool_d3_s11_no2: best iteration 798, fit 172,888 rows (21,252 positive), 23.0s
  trained N0_A_pooled_pool_d3_s11_no3: best iteration 799, fit 174,598 rows (22,962 positive), 21.3s
  trained N0_A_pooled_pool_d3_s11_no4: best iteration 799, fit 168,752 rows (17,116 positive), 21.7s
  trained N0_A_pooled_pool_d3_s11_no5: best iteration 794, fit 168,716 rows (17,080 positive), 22.1s
  trained N0_A_pooled_pool_d3_s11_no6: best iteration 796, fit 172,526 rows (20,890 positive), 22.7s
  trained N1_A_pooled_pool_d3_s22: best iteration 799, fit 175,506 rows (23,870 positive), 23.1s
  trained N1_A_pool

## 7 · VAL-3 evaluation, chance baseline, LOFO, unseen families

In [ ]:
if FIN_DONE:
    def nanw(a): a = a.astype('float64').copy(); a[:B.WARM] = np.nan; return a
    def ae_fn(mode): return lambda s, key: nanw(SCO[(s, mode)][key][:, 0])
    def rz_fn(mode): return lambda s, key: (XOF(s, mode, key), nanw(RZ[(s, mode, key)]))[1]
    FIN = {m: Model(m, *SELECTED[m], SEEDS) for m in MODES}; OUTROWS = []; HITS = {}
    for mode in MODES:
        comps = {'XGBoost v2': FIN[mode].score, 'AE-ctx fmax': ae_fn(mode), 'robust-z': rz_fn(mode)}
        for mname, fn in comps.items():
            for bud, rule in [(b_, 'T1') for b_ in BUDGETS] + [(PB, 'T0')]:
                dfh, dev = val_hits(fn, rule, bud)
                if rule == 'T1': HITS[(mode, mname, bud)] = dfh
                po = dfh[dfh.fam.isin(OLD)].hit.to_numpy(); pn = dfh[dfh.fam.isin(NEW)].hit.to_numpy()
                OUTROWS.append({'norm': mode, 'model': mname, 'thr': rule, 'budget/day': bud, 'DEV genuine eps/day': float(np.mean(list(dev.values()))), 'VAL-3 recall 6 old': macro_rate(dfh, OLD), 'old 95% CI': tuple(np.round(B.bootstrap_ci(po), 3)),
                                'VAL-3 recall 2 new': macro_rate(dfh, NEW), 'new 95% CI': tuple(np.round(B.bootstrap_ci(pn), 3))})
    VT = pd.DataFrame(OUTROWS); VT.to_csv(OUT/'xgb_val_table.csv', index=False); say('--- VAL-3 recall (T1 = deployable rolling threshold, T0 = static threshold, shown for 2/day only; pooled-event bootstrap CI) ---'); say(VT.round(3).to_string(index=False))
    # chance baseline: the model's own genuine DEV alarms placed at random times (circular shifts), primary budget
    say('--- chance baseline at 2 episodes/day (same alarms, random time) ---'); CH = {}
    for mode in MODES:
        for mname, fn in {'XGBoost v2': FIN[mode].score, 'AE-ctx fmax': ae_fn(mode)}.items():
            ro, rn = [], []
            for s in SYMBOLS:
                g = fn(s, 'GEN_L1').astype('float64'); par = B.t1_q(g, L1V, PB); a_dev = B.alarms(g, 'T1', par)[90720:]; evs = pd.concat([ev for r, (f, ev) in D[s]['sets']['VAL'].items()], ignore_index=True)
                ro.append(chance_recall(a_dev, evs, OLD)); rn.append(chance_recall(a_dev, evs, NEW))
            CH[(mode, mname)] = (float(np.mean(ro)), float(np.mean(rn)))
            say(f'{mode} {mname}: chance recall 6 old {np.mean(ro):.3f}, 2 new {np.mean(rn):.3f}')
    # LOFO (6 old families; the held-out family is in neither the fit nor the early-stopping rows)
    LR = []
    for mode in MODES:
        for fam in OLD:
            ml = Model(mode, *SELECTED[mode], [11], lofo=fam); dl, _ = val_hits(ml.score, 'T1', PB); da = HITS[(mode, 'AE-ctx fmax', PB)]; df_ = HITS[(mode, 'XGBoost v2', PB)]; dz = HITS[(mode, 'robust-z', PB)]
            q = lambda d_: float(d_[d_.fam == fam].hit.mean())
            LR.append({'norm': mode, 'family': B.FAMS[fam], 'events': int((dl.fam == fam).sum()), 'XGB without this family': q(dl), 'XGB trained on it': q(df_), 'AE-ctx fmax': q(da), 'robust-z': q(dz)})
    LT = pd.DataFrame(LR); LT.to_csv(OUT/'xgb_lofo_table.csv', index=False); say('--- leave-one-family-out, VAL-3 events of the held-out family, 2 episodes/day, T1 ---'); say(LT.round(3).to_string(index=False))
    LM = LT.groupby('norm')[['XGB without this family', 'XGB trained on it', 'AE-ctx fmax', 'robust-z']].mean(); say('LOFO macro over the 6 families:'); say(LM.round(3).to_string())
    # per family on the 8 families at the primary budget (final models)
    PF = []
    for mode in MODES:
        for mname in ['XGBoost v2', 'AE-ctx fmax', 'robust-z']:
            d_ = HITS[(mode, mname, PB)]; r_ = d_.groupby('fam').hit.agg(['mean', 'sum', 'count'])
            for fam, row in r_.iterrows(): PF.append({'norm': mode, 'model': mname, 'family': B.FAMS[int(fam)], 'recall': row['mean'], 'detected': int(row['sum']), 'events': int(row['count'])})
    PFT = pd.DataFrame(PF); PFT.to_csv(OUT/'xgb_val_by_family.csv', index=False); say('--- per family, 2 episodes/day, T1 ---'); say(PFT.pivot_table(index='family', columns=['norm', 'model'], values='recall').round(2).to_string())
    # indicative H-B5 on VAL-3 (the formal test is on TEST-3 in notebook 32)
    HB5 = {}
    for mode in MODES:
        x = macro_rate(HITS[(mode, 'XGBoost v2', PB)], NEW); z = macro_rate(HITS[(mode, 'robust-z', PB)], NEW); a = macro_rate(HITS[(mode, 'AE-ctx fmax', PB)], NEW)
        HB5[mode] = {'XGB new families': x, 'AE-ctx new families': a, 'robust-z new families': z, 'best trained minus rz': max(x, a) - z, 'indicative H-B5 (>= 0.05)': bool(max(x, a) - z >= 0.05)}
        say(f'{mode}: unseen families recall (VAL-3): XGBoost {x:.3f}, AE-ctx {a:.3f}, robust-z {z:.3f}; best trained minus rz {max(x, a) - z:+.3f} -> indicative H-B5 {max(x, a) - z >= 0.05}')

--- VAL-3 recall (T1 = deployable rolling threshold, T0 = static threshold, shown for 2/day only; pooled-event bootstrap CI) ---
norm       model thr  budget/day  DEV genuine eps/day  VAL-3 recall 6 old     old 95% CI  VAL-3 recall 2 new     new 95% CI
  N0  XGBoost v2  T1           1                0.815               0.101 (0.075, 0.129)               0.056  (0.021, 0.09)
  N0  XGBoost v2  T1           2                1.500               0.134 (0.101, 0.167)               0.056  (0.021, 0.09)
  N0  XGBoost v2  T1           4                2.963               0.184 (0.149, 0.221)               0.097 (0.049, 0.146)
  N0  XGBoost v2  T0           2                1.648               0.689  (0.643, 0.73)               0.792 (0.722, 0.854)
  N0 AE-ctx fmax  T1           1                0.815               0.112  (0.083, 0.14)               0.062 (0.028, 0.104)
  N0 AE-ctx fmax  T1           2                1.389               0.145 (0.114, 0.175)               0.083 (0.042, 0.132)
  N

## 8 · Save scores, TreeSHAP, decisions

In [ ]:
if FIN_DONE:
    # scores of every series (final models, both normalisations) for notebook 31 / 32; CAL-3 and genuine Aug are scored here but not looked at
    for mode in MODES:
        for s in SYMBOLS:
            keys = ['GEN_L1', 'GEN_AO'] + [f'{n_}_r{r}' for n_ in ['TRAIN', 'VAL', 'CAL'] for r in D[s]['sets'][n_]]
            np.savez_compressed(OUT/'scores'/f'xgb_v2_{s}_{mode}.npz', **{k: FIN[mode].score(s, k) for k in keys})
            for k in keys:
                if k.startswith(('CAL', 'GEN_AO')): XS.pop((s, mode, k), None)
    say('scores saved: models_b2_v1/scores/xgb_v2_{symbol}_{N0|N1}.npz')
    # exact TreeSHAP on VAL-3 positive bars (old and new families) + 3000 clean bars per symbol, mean of the seeds
    GROUPS = FEATS + ['cross', 'ae']
    def group_of(n_): return n_.split('__')[0] if n_.split('__')[0] in FEATS else ('cross' if n_.startswith('x__') else 'ae')
    SHAP = {}; rgs = np.random.default_rng(5)
    for mode in MODES:
        fs = SELECTED[mode][0]; nm = NAMES[fs]; gi = np.array([GROUPS.index(group_of(n_)) for n_ in nm]); imp_rows = []; fam_rows = []; maxerr = 0.0
        for s in SYMBOLS:
            P, F_, X_ = [], [], []
            for r in D[s]['sets']['VAL']:
                lab, inseg, clean, pos = RS[(s, 'VAL', r)]; ii = np.flatnonzero(pos)
                if len(ii) > 4000: ii = np.sort(rgs.choice(ii, 4000, replace=False))
                X_.append(XOF(s, mode, f'VAL_r{r}')[ii][:, :NCOL[fs]]); F_.append(lab[ii])
                if r == 0:
                    jj = np.flatnonzero(clean); jj = np.sort(rgs.choice(jj, min(3000, len(jj)), replace=False)); X_.append(XOF(s, mode, 'VAL_r0')[jj][:, :NCOL[fs]]); F_.append(np.zeros(len(jj), int))
            X_ = np.concatenate(X_); F_ = np.concatenate(F_); dm = xgb.DMatrix(X_, feature_names=nm); cs = []
            for b_, bi in FIN[mode].boosters(s):
                c_ = b_.predict(dm, pred_contribs=True, iteration_range=(0, bi + 1)); mg = b_.predict(dm, output_margin=True, iteration_range=(0, bi + 1)); maxerr = max(maxerr, float(np.abs(c_.sum(axis=1) - mg).max())); cs.append(c_[:, :-1])
            C_ = np.mean(cs, axis=0); G = np.zeros((len(C_), len(GROUPS)))
            for g_ in range(len(GROUPS)): G[:, g_] = np.abs(C_[:, gi == g_]).sum(axis=1)
            for fam in range(0, 9):
                if (F_ == fam).any(): fam_rows.append(dict(zip(['sym', 'family'] + GROUPS, [s, 'clean' if fam == 0 else B.FAMS[fam]] + G[F_ == fam].mean(axis=0).tolist())))
            imp_rows.append(pd.Series(np.abs(C_[F_ > 0]).mean(axis=0), index=nm))
        imp = pd.concat(imp_rows, axis=1).mean(axis=1).sort_values(ascending=False); fam_df = pd.DataFrame(fam_rows).groupby('family')[GROUPS].mean()
        SHAP[mode] = {'additivity_max_error': maxerr, 'top_features_positive': [(k, float(v)) for k, v in imp.head(12).items()], 'by_family_group_mean_abs': fam_df.round(4).to_dict('index')}
        assert maxerr < 1e-3, 'TreeSHAP additivity'
        say(f'{mode} SHAP (selected {cname(SELECTED[mode])}): additivity max error {maxerr:.1e}; top features on positive bars: ' + ', '.join(f'{k} {v:.2f}' for k, v in imp.head(6).items())); say(fam_df.round(2).to_string())
    say('PASS T10 TreeSHAP contributions add up to the model margin')
    DEC = {'selected': {m: cname(SELECTED[m]) for m in MODES}, 'H-B4': HB4, 'H-B5 indicative on VAL-3': HB5, 'chance': {f'{k[0]}|{k[1]}': v for k, v in CH.items()}, 'lofo_macro': LM.round(4).to_dict('index'),
           'shap': SHAP, 'note': 'N0/N1 adoption for XGBoost (H-B1/H-B2 on CAL) and CAL-3 recall are decided in notebook 31; TEST is not touched.'}
    (OUT/'xgb_v2_decisions.json').write_text(json.dumps(DEC, indent=1, default=str)); say('saved xgb_v2_decisions.json, xgb_selection_table.csv, xgb_val_table.csv, xgb_lofo_table.csv, xgb_val_by_family.csv')

scores saved: models_b2_v1/scores/xgb_v2_{symbol}_{N0|N1}.npz
N0 SHAP (selected A|pooled|d3): additivity max error 1.5e-05; top features on positive bars: trade_flow_imbalance__amax15 1.10, trade_flow_imbalance__std15 0.51, log_returns__mean15 0.38, trade_flow_imbalance__amax3 0.36, log_returns__amax15 0.33, trade_flow_imbalance__mean15 0.33
                           log_returns  trade_flow_imbalance  hl_spread_ratio  rush_order_density  volume_acceleration  large_trade_count  cross   ae
family                                                                                                                                               
abnormal_volume_burst             1.38                  0.94             0.91                0.52                 0.61               0.74   0.60  0.0
clean                             1.11                  0.97             0.81                0.48                 0.55               0.42   0.41  0.0
fake_breakout_like                1.81                  

## 9 · Share block

In [ ]:
block = '\n'.join(LINES); (OUT/'b2_step3_share_block.txt').write_text(block)
tail = block[-14000:]; tail = tail[tail.find('\n') + 1:] if len(block) > 14000 else block
print('\n===== SHARE BLOCK (best: attach the file models_b2_v1/b2_step3_share_block.txt; otherwise copy everything below) =====\n' + tail)


===== SHARE BLOCK (best: attach the file models_b2_v1/b2_step3_share_block.txt; otherwise copy everything below) =====
  trained N1_A_per_symbol_ETHUSDT_d5_s11: best iteration 799, fit 87,791 rows (11,917 positive), 19.3s
N1 A|per_symbol|d5    VAL-3 recall(old6,T1) 0.145  T0 0.636  PR-AUC 0.756  DEV eps/day 1.63
  trained N1_B_pooled_pool_d3_s11: best iteration 798, fit 175,506 rows (23,870 positive), 27.4s
N1 B|pooled|d3        VAL-3 recall(old6,T1) 0.143  T0 0.627  PR-AUC 0.752  DEV eps/day 1.81
  trained N1_B_pooled_pool_d5_s11: best iteration 795, fit 175,506 rows (23,870 positive), 34.6s
N1 B|pooled|d5        VAL-3 recall(old6,T1) 0.134  T0 0.673  PR-AUC 0.776  DEV eps/day 1.59
  trained N1_B_per_symbol_BTCUSDT_d3_s11: best iteration 795, fit 87,715 rows (11,953 positive), 14.3s
  trained N1_B_per_symbol_ETHUSDT_d3_s11: best iteration 793, fit 87,791 rows (11,917 positive), 14.5s
N1 B|per_symbol|d3    VAL-3 recall(old6,T1) 0.112  T0 0.627  PR-AUC 0.747  DEV eps/day 1.13
  trained